# Validation of the resources needed per each component

$\newcommand{\ket}[1]{\left|#1\right\rangle} \newcommand{\bra}[1]{\left\langle #1\right|} \newcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle} \newcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}
$This notebook serves as to validate numerically the analysis of the resources (qubits, depth) in the previous documents. 

In [14]:
# allows us to have visibility on our package without installing it in editing mode
import sys;
if ".." not in sys.path: sys.path.append("..")
import numpy as np
from qiskit.circuit import QuantumCircuit, Gate
from monaqa2.qiskit.utils_qiskit import get_nc_depth, qiskit_to_clifford_rz

def validate_resource(gate: Gate, expected_num_qubits: int, expected_depth: int):
    # print(f"Checking {gate.name}")
    # print(f"  qubits: actual={gate.num_qubits}, expected={expected_num_qubits}")
    if gate.num_qubits != expected_num_qubits:
        raise ValueError(f"Class {gate.name} has {gate.num_qubits} qubits, expected {expected_num_qubits}")
    qc = QuantumCircuit(expected_num_qubits)
    qc.append(gate, range(expected_num_qubits))
    actual_depth = get_nc_depth(qc)
    # print(f"  non-Clifford depth: actual={actual_depth}, upper_bound={expected_depth}")
    if actual_depth > expected_depth:
        raise ValueError(f"Class {gate.name} has depth {actual_depth}, upper bound was {expected_depth}")
    # print(f"  OK\n")

## Primitives

### Reflections

Method synth_mcx_2_clean_kg24 scales as $14 \lceil \log_2(m) \rceil - 13$ for all $m \ge 3$. 

In [3]:
from qiskit.synthesis.multi_controlled.mcx_synthesis import synth_mcx_2_clean_kg24

# for m in list(range(3, 16)) + [(2**n) for n in range(4, 16)]:
#     qc = QuantumCircuit(m+3)
#     qc.append(synth_mcx_2_clean_kg24(m), range(m+3))
#     expected_depth = int(14*np.ceil(np.log2(m))-13)
#     validate_resource(qc, m+3, expected_depth)

## Arithmetic

### Hybrid arithmetic

In [4]:
from monaqa2.qiskit.arithmetic_fully_phase import GivensRotation, ControlledGivensRotation, ReflectionZero, ControlledReflectionZero
from monaqa2.qiskit.arithmetic_hybrid import ThreeTwoCompressor, Majority, WallaceTreeAdder, ConditionalTermsLoader, DeltaEnergy, PrepareOneBodyHamiltonian, SelectOneBodyHamiltonian, ControlledSelectOneBodyHamiltonian, QubitizedOneBodyHamiltonian, ControlledQubitizedOneBodyHamiltonian, CutoffTail, SqrtExpArithmetic, PositivePartSelector, HybridPhaseArithmetic


def validate_hybrid_arithmetic_resources(M: int, W: int, m: int, d: int):
    n = int((np.sqrt(1 + 8 * M) - 1) / 2)
    if n * (n + 1) // 2 != M:
        raise ValueError(f"{M=} is not of the form n + binom(n,2)")
    if not (3 <= m <= W - 1):
        raise ValueError(f"Expected active tail with 3 <= m <= W-1, got {m=} and {W=}")

    F = W - 1
    ell_M, ell_W, ell_m = int(np.ceil(np.log2(M))), int(np.ceil(np.log2(W))), int(np.ceil(np.log2(m)))

    h = np.zeros(n)
    h[0] = 0.5
    J = np.zeros((n, n))
    h_signal = np.ones(W)
    alpha = ConditionalTermsLoader._alpha(h, J)
    eps = float(np.exp(-1.0))
    beta = float(2 ** (m + 1))

    expected_num_qubit_givens = 2
    expected_depth_givens = 2
    validate_resource(GivensRotation(0.25, 0.75), expected_num_qubit_givens, expected_depth_givens)

    expected_num_qubit_c_givens = 3
    expected_depth_c_givens = 6
    validate_resource(ControlledGivensRotation(0.25, 0.75), expected_num_qubit_c_givens, expected_depth_c_givens)

    expected_num_qubit_loader = M * W + 2 * M - n
    expected_depth_loader = 0
    validate_resource(ConditionalTermsLoader(n, h, J, F, invert_coefficients=False), expected_num_qubit_loader, expected_depth_loader)

    expected_num_qubit_compressor = 5
    expected_depth_compressor = 3
    validate_resource(ThreeTwoCompressor(), expected_num_qubit_compressor, expected_depth_compressor)

    expected_num_qubit_wallace = 6 * W * M - 2 * W - 2 * M + 1
    expected_depth_wallace = 6 * ell_M + 6 * W + 12
    validate_resource(WallaceTreeAdder(2 * M, W), expected_num_qubit_wallace, expected_depth_wallace)

    expected_num_qubit_delta = 2 * n + 6 * W * M - 2 * W - 2 * M + 1
    expected_depth_delta = 6 * ell_M + 6 * W + 12
    validate_resource(DeltaEnergy(n, h, J, F), expected_num_qubit_delta, expected_depth_delta)

    expected_num_qubit_prepare = W
    expected_depth_prepare = 2 * ell_W
    validate_resource(PrepareOneBodyHamiltonian(W, h_signal), expected_num_qubit_prepare, expected_depth_prepare)

    expected_num_qubit_select = 2 * W
    expected_depth_select = 0
    validate_resource(SelectOneBodyHamiltonian(W, h_signal), expected_num_qubit_select, expected_depth_select)

    expected_num_qubit_c_select = 3 * W
    expected_depth_c_select = 1
    validate_resource(ControlledSelectOneBodyHamiltonian(W, h_signal), expected_num_qubit_c_select, expected_depth_c_select)

    expected_num_qubit_reflection = W + 2
    expected_depth_reflection = 14 * ell_W - 13
    validate_resource(ReflectionZero(W), expected_num_qubit_reflection, expected_depth_reflection)

    expected_num_qubit_c_reflection = W + 3
    expected_depth_c_reflection = 14 * ell_W - 13
    validate_resource(ControlledReflectionZero(W), expected_num_qubit_c_reflection, expected_depth_c_reflection)

    expected_num_qubit_qubitized = 2 * W + 2
    expected_depth_qubitized = 18 * ell_W - 13
    validate_resource(QubitizedOneBodyHamiltonian(W, h_signal), expected_num_qubit_qubitized, expected_depth_qubitized)

    expected_num_qubit_c_qubitized = 3 * W + 2
    expected_depth_c_qubitized = 18 * ell_W - 12
    validate_resource(ControlledQubitizedOneBodyHamiltonian(W, h_signal), expected_num_qubit_c_qubitized, expected_depth_c_qubitized)

    expected_num_qubit_sqrt_exp = 3 * W + 2
    expected_depth_sqrt_exp = d * (54 * ell_W - 31) + 3
    validate_resource(SqrtExpArithmetic(W, beta, alpha, eps, eps_tail=eps, degree=d), expected_num_qubit_sqrt_exp, expected_depth_sqrt_exp)

    expected_num_qubit_positive = 3 * W
    expected_depth_positive = 3
    validate_resource(PositivePartSelector(W), expected_num_qubit_positive, expected_depth_positive)

    expected_num_qubit_cutoff = 2 * W + 3
    expected_depth_cutoff = 14 * ell_m + W - m - 14
    validate_resource(CutoffTail(W, m), expected_num_qubit_cutoff, expected_depth_cutoff)

    expected_num_qubit_hybrid = 2 * n + 6 * W * M - 2 * M + 2
    expected_depth_hybrid = d * (54 * ell_W - 31) + 12 * (ell_M + 1) + 14 * W - 2 * m + 28 * ell_m - 25
    validate_resource(HybridPhaseArithmetic(n, h, J, F, beta, eps, degree=d), expected_num_qubit_hybrid, expected_depth_hybrid)

    print(f"All checks passed for {n=}, {M=}, {W=}, {m=}, {d=}")

In [5]:
def triangular_M(n: int) -> int:
    return n + n * (n - 1) // 2

# Minimal smoke tests.
validate_hybrid_arithmetic_resources(M=triangular_M(2), W=5, m=3, d=1)
validate_hybrid_arithmetic_resources(M=triangular_M(3), W=6, m=3, d=1)
validate_hybrid_arithmetic_resources(M=triangular_M(4), W=7, m=3, d=2)
validate_hybrid_arithmetic_resources(M=triangular_M(5), W=8, m=4, d=2)
# More meaningful small/medium tests: non-power-of-two W, different m, different d.
validate_hybrid_arithmetic_resources(M=triangular_M(5), W=9, m=4, d=3)
validate_hybrid_arithmetic_resources(M=triangular_M(6), W=10, m=4, d=3)
validate_hybrid_arithmetic_resources(M=triangular_M(7), W=11, m=5, d=4)
validate_hybrid_arithmetic_resources(M=triangular_M(8), W=12, m=5, d=4)
# Stress tests for the Wallace tree and cutoff tail, still not insane.
validate_hybrid_arithmetic_resources(M=triangular_M(10), W=12, m=5, d=5)
validate_hybrid_arithmetic_resources(M=triangular_M(12), W=14, m=6, d=6)
validate_hybrid_arithmetic_resources(M=triangular_M(15), W=16, m=7, d=8)
# Tail edge cases: smallest active m and largest allowed m.
validate_hybrid_arithmetic_resources(M=triangular_M(6), W=9, m=3, d=3)
validate_hybrid_arithmetic_resources(M=triangular_M(6), W=9, m=8, d=3)

All checks passed for n=2, M=3, W=5, m=3, d=1
All checks passed for n=3, M=6, W=6, m=3, d=1
All checks passed for n=4, M=10, W=7, m=3, d=2
All checks passed for n=5, M=15, W=8, m=4, d=2
All checks passed for n=5, M=15, W=9, m=4, d=3
All checks passed for n=6, M=21, W=10, m=4, d=3
All checks passed for n=7, M=28, W=11, m=5, d=4
All checks passed for n=8, M=36, W=12, m=5, d=4
All checks passed for n=10, M=55, W=12, m=5, d=5
All checks passed for n=12, M=78, W=14, m=6, d=6
All checks passed for n=15, M=120, W=16, m=7, d=8
All checks passed for n=6, M=21, W=9, m=3, d=3
All checks passed for n=6, M=21, W=9, m=8, d=3


## Proposals

In [2]:
import numpy as np
from monaqa2.qiskit.proposal_uniform import ProposalUniform
from monaqa2.qiskit.proposal_local import ProposalLocal
from monaqa2.qiskit.proposal_qemc import ProposalQemc


def validate_proposal_resources(n: int):
    if n < 2:
        raise ValueError("Use n >= 2.")

    ell_n = int(np.ceil(np.log2(n)))
    num_trotter_steps = 50

    h = np.ones(n)
    J = np.ones((n, n)) - np.eye(n)
    J = np.triu(J, 1) + np.triu(J, 1).T
    gamma = np.ones(n)
    t = 1.0

    expected_num_qubit_uniform = 2 * n
    expected_depth_uniform = 0
    validate_resource(ProposalUniform(n), expected_num_qubit_uniform, expected_depth_uniform)

    expected_num_qubit_local = 2 * n
    expected_depth_local = 13 * ell_n + 15
    validate_resource(ProposalLocal(n, k=1), expected_num_qubit_local, expected_depth_local)

    n_matching_rounds = n if n % 2 == 1 else n - 1
    expected_num_qubit_qemc = 2 * n
    expected_depth_qemc = num_trotter_steps * (n_matching_rounds + 3)
    validate_resource(ProposalQemc(n, h, J, gamma, t, num_trotter_steps=num_trotter_steps), expected_num_qubit_qemc, expected_depth_qemc)

In [8]:
for i in list(range(2, 16)) + [2**n for n in range(4, 7+1)]:
    validate_proposal_resources(i)
    print(f"{i=:3d} ok | ", end="")

i=  2 ok | i=  3 ok | i=  4 ok | i=  5 ok | i=  6 ok | i=  7 ok | i=  8 ok | i=  9 ok | i= 10 ok | i= 11 ok | i= 12 ok | i= 13 ok | i= 14 ok | i= 15 ok | i= 16 ok | i= 32 ok | i= 64 ok | 

## Accept path - Reflection

In [15]:
import numpy as np
from monaqa2.qiskit.reflection import Reflection
from monaqa2.qiskit.accept_path import AcceptPath


def validate_reflection_accept_path_resources(n: int, c: int):
    if n < 3:
        raise ValueError("Use n >= 3.")
    if c < 1:
        raise ValueError("Use c >= 1.")

    q_reflection = n + c - 1

    expected_num_qubit_reflection = 2 * n + c + (2 if q_reflection > 2 else 0)
    expected_depth_reflection = 14 * int(np.ceil(np.log2(q_reflection))) - 13 if q_reflection > 2 else (1 if q_reflection == 2 else 0)
    validate_resource(Reflection(n, coins=c), expected_num_qubit_reflection, expected_depth_reflection)

    expected_num_qubit_accept = 2 * n + c + 1 + max(n - 1, 2 if c > 2 else 0)
    expected_depth_accept = 28 * int(np.ceil(np.log2(c))) - 23 if c > 2 else 3
    validate_resource(AcceptPath(n, coins=c), expected_num_qubit_accept, expected_depth_accept)


def validate_reflection_accept_path_resources_hybrid(n: int, S: int):
    if n < 3:
        raise ValueError("Use n >= 3.")
    if S < 1:
        raise ValueError("Use S >= 1.")

    c = 4 + 7 * S
    validate_reflection_accept_path_resources(n, c)

In [16]:
validate_reflection_accept_path_resources(n=8, c=32)
validate_reflection_accept_path_resources_hybrid(n=8, S=4)
validate_reflection_accept_path_resources_hybrid(n=16, S=6)